# Patient P3 — Session Selection and Preprocessing

## Objective

This notebook evaluates raw respiratory session files for Patient P3 and identifies physiologically suitable treatment sessions for inclusion in the variability analysis pipeline.

In [1]:
import os
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

from private_patient_mapping import public_filename

## Raw Respiratory Signal Loading

Functions for parsing ABC respiratory waveform `.dat` files and extracting usable signal data.

In [2]:
def load_raw(file_path):

    col_names = [
        "Time",
        "Volume",
        "Balloon_Valve",
        "Patient_Switch",
        "Gating_Mode",
        "Gating_Status",
        "Relay_State"
    ]

    start = None

    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if "HeaderEnd" in line:
                start = i + 1
                break

    # Skip files without valid ABC header
    if start is None:
        print(f"Skipping (no HeaderEnd): {public_filename(file_path)}")
        return None

    try:
        df = pd.read_csv(
            file_path,
            sep=r'\s*;\s*',
            skiprows=start,
            names=col_names,
            engine="python",
            na_values=["-", " - "]
        )
        return df

    except Exception as e:
        print(f"Error reading {public_filename(file_path)}: {e}")
        return None

## Signal Extraction and Cleaning

In [3]:
def extract_signal(df):
    df = df.dropna(subset=["Time", "Volume"])

    df["Time"] = df["Time"].astype(float)
    df["Volume"] = df["Volume"].astype(float)

    return df

## Respiratory Cycle Detection

Peak and trough detection is used to estimate physiological respiratory cycles from waveform data.

In [4]:
def detect_cycles(time, volume):

    # Detect peaks (inhale)
    peaks, _ = find_peaks(volume, distance=10)

    # Detect troughs (exhale)
    troughs, _ = find_peaks(-volume, distance=10)

    extrema = np.sort(np.concatenate([peaks, troughs]))

    cycles = []

    for i in range(len(extrema) - 2):
        i1, i2, i3 = extrema[i], extrema[i+1], extrema[i+2]

        duration = time[i3] - time[i1]
        amplitude = abs(volume[i2] - volume[i1])

        # Basic validity filter
        if duration > 0:
            cycles.append((duration, amplitude))

    return cycles

## Session Quality Evaluation

Sessions are filtered using duration and cycle-count criteria to exclude physiologically insufficient recordings.

In [5]:
def evaluate_signal(df):

    df = extract_signal(df)

    if len(df) < 50:
        return None

    time = df["Time"].values
    volume = df["Volume"].values

    duration = time[-1] - time[0]
    volume_std = np.std(volume)
    cycles = detect_cycles(time, volume)
    num_cycles = len(cycles)

    return {
        "duration": duration,
        "volume_std": volume_std,
        "num_cycles": num_cycles
    }

def evaluate_file_full(file_path):

    df = load_raw(file_path)

    if df is None:
        print(f"Rejected(no header): {public_filename(file_path)}")
        return None

    signal = evaluate_signal(df)

    if signal is None:
        print(f"Rejected(weak/short signal): {public_filename(file_path)}")
        return None

    # Physiological session inclusion criteria
    if signal["duration"] < 300:
        print(f"Rejected(too short): {public_filename(file_path)}")
        return None

    if signal["num_cycles"] < 100:
        print(f"Rejected(too few cycles): {public_filename(file_path)}")
        return None

    if signal["volume_std"] < 0.2:
        print(f"Rejected(low variability) {public_filename(file_path)}")
        return None

    print(f"Selected: {public_filename(file_path)}")

    return {
        "file": public_filename(file_path),
        "signal": signal
    }

## Session Selection Execution

In [6]:
def select_best_files(folder_path):
    selected = []

    for file in os.listdir(folder_path):
        if file.endswith(".dat"):
            path = os.path.join(folder_path, file)
            res = evaluate_file_full(path)

            if res:
                selected.append(res)

    return selected

select_best_files(".")

Selected: P3_ct.dat
Selected: P3_SIMULATION.dat
Selected: P3_tx-10.dat
Rejected(low variability) P3_tx-11.dat
Selected: P3_tx-11_7_25_2024 12_05_42 PM.dat
Selected: P3_tx-12.dat
Selected: P3_tx-13.dat
Selected: P3_tx-14.dat
Selected: P3_tx-15.dat
Selected: P3_tx-16.dat
Selected: P3_tx-17.dat
Selected: P3_tx-17_8_5_2024 12_03_46 PM.dat
Selected: P3_tx-18.dat
Selected: P3_tx-19.dat
Selected: P3_tx-5.dat
Selected: P3_tx-6.dat
Selected: P3_tx-7.dat
Selected: P3_tx-8.dat
Selected: P3_tx-9.dat
Selected: P3_tx1.dat
Selected: P3_tx2.dat
Selected: P3_tx3.dat
Selected: P3_tx4.dat


[{'file': 'P3_ct.dat',
  'signal': {'duration': np.float64(876.26),
   'volume_std': np.float64(0.3215221149969281),
   'num_cycles': 393}},
 {'file': 'P3_SIMULATION.dat',
  'signal': {'duration': np.float64(717.58),
   'volume_std': np.float64(0.28865631379727646),
   'num_cycles': 284}},
 {'file': 'P3_tx-10.dat',
  'signal': {'duration': np.float64(2853.1400000000003),
   'volume_std': np.float64(0.2586135187678339),
   'num_cycles': 597}},
 {'file': 'P3_tx-11_7_25_2024 12_05_42 PM.dat',
  'signal': {'duration': np.float64(820.74),
   'volume_std': np.float64(0.34576842830383564),
   'num_cycles': 373}},
 {'file': 'P3_tx-12.dat',
  'signal': {'duration': np.float64(1245.3999999999999),
   'volume_std': np.float64(0.3072553373452874),
   'num_cycles': 539}},
 {'file': 'P3_tx-13.dat',
  'signal': {'duration': np.float64(1085.12),
   'volume_std': np.float64(0.3528937721182875),
   'num_cycles': 509}},
 {'file': 'P3_tx-14.dat',
  'signal': {'duration': np.float64(1010.74),
   'volume_st

## Final Session Selection

Sessions were screened using duration and respiratory cycle-count criteria
to exclude short or physiologically insufficient recordings. Additional
consistency validation was performed to identify duplicate or repeated
waveform files.

For Patient P3, six treatment sessions were retained for downstream
variability analysis:
- P3_tx-4.dat   
- P3_tx-5.dat   
- P3_tx-7.dat   
- P3_tx-6.dat   
- P3_tx3.dat    
- P3_tx-10.dat

## Duplicate and Consistency Validation

Additional validation checks are performed to identify:
- duplicate respiratory recordings,
- filename inconsistencies,
- and repeated waveform sessions.

These checks were used to improve preprocessing reliability
before session selection.

In [7]:
def compute_signature(df):
    df = extract_signal(df)

    if len(df) < 10:
        return None

    time = df["Time"].values
    volume = df["Volume"].values

    return {
        "start_time": round(time[0], 2),
        "end_time": round(time[-1], 2),
        "length": len(df),
        "mean_vol": round(np.mean(volume), 4),
        "std_vol": round(np.std(volume), 4)
    }

def signature_key(sig):
    return (
        sig["start_time"],
        sig["end_time"],
        sig["length"],
        sig["mean_vol"],
        sig["std_vol"]
    )

def analyze_folder_consistency(folder_path):
    signatures = {}
    name_map = {}

    for file in os.listdir(folder_path):
        if not file.endswith(".dat"):
            continue

        path = os.path.join(folder_path, file)
        df = load_raw(path)

        if df is None:
            continue

        sig = compute_signature(df)
        if sig is None:
            continue

        key = signature_key(sig)

        # Check for identical respiratory recordings
        if key in signatures:
            print("Duplicate session detected:")
            print(f"   {public_filename(file)} == {public_filename(signatures[key])}")
        else:
            signatures[key] = file

        # Check for same filename consistency
        base_name = file.split(".dat")[0]

        if base_name in name_map:
            prev_sig = name_map[base_name]

            if signature_key(prev_sig) != key:
                print("Filename inconsistency detected:")
                print(f"   {base_name}")
        else:
            name_map[base_name] = sig

    print("\nFolder consistency check completed.")

In [8]:
analyze_folder_consistency(".")


Folder consistency check completed.


## Preprocessing Outcome

The selected treatment sessions passed duration, cycle-count,
and consistency validation checks and were retained for
downstream respiratory variability analysis.